# Dijet JEU systematic uncertainty

Compare the JEU Up, Down, and default reconstructed dijet pseudorapidity distributions in MinimumBias, Jet60, Jet80, and Jet100 data. Full CM distributions are normalized to unit integral before comparison. Forward/Backward distributions are left unnormalized; their ratios always use standard independent-error propagation, never ROOT's binomial option.

Dedicated variation/default plots provide the signed shape variations used to estimate the JEU systematic uncertainty.

## Environment and imports

This notebook locates the repository dynamically and imports PyROOT from the
active project environment. Start Jupyter from the repository root with
`py-env/bin/python -m jupyter notebook`; no machine-specific ROOT paths are
added at runtime.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from dataclasses import replace
import math
import os

import sys

# Locate the repository without relying on a machine-specific absolute path.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "CMakeLists.txt").is_file()
        and (candidate / "hist_analysis").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Cannot locate the jetAnalysis repository. Start Jupyter from its root."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.python.notebook_setup import load_root

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import (
    DIJET_DELTA_PHI_SELECTION_LABEL,
    STANDARD_DIJET_ETA_CUT_INDEX,
)
from hist_analysis.python.dijet_closures import (
    DijetClosureCurve, build_dijet_gen_comparisons,
)
from hist_analysis.python.histogram_ops import ratio_to_nominal
from hist_analysis.python.histogram_io import resolve_data_file
from hist_analysis.python.plotting import draw_overlay
from hist_analysis.python.root_style import COLORS, DEFAULT_PLOT_STYLE, save_canvas
from hist_analysis.python.systematic_fits import (
    calculate_bin_by_bin_systematic, fit_histogram_variations,
    format_fit_summary_lines, smooth_systematic_running_max,
)


In [ ]:
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

## Configuration

`DATA_DIRECTION` selects `combined`, `Pbgoing`, or `pgoing` data for MinimumBias, Jet60, Jet80, and Jet100. `DATA_SELECTION` selects the matching `jetId`, `trkMax`, or `noSel` production through the shared data-output filename resolver.

`PTAVE_BINS` independently selects the intervals analyzed for each trigger sample, following `06_data_check.ipynb`. `FULL_COMPARISON_RATIO_OPTION` controls errors on normalized Up/Def and Down/Def shape ratios. `FB_COMPARISON_RATIO_OPTION` controls errors only on the later ratio of two already-constructed F/B histograms. Set either to `''` for ROOT's standard propagation or `'B'` for option B. The construction of every F/B histogram is hard-coded to `''`. Full-distribution and F/B comparison ratios have separately configurable fit functions and initial parameters.

In [ ]:
DATA_DIR = Path(os.environ.get('PPB_DATA_DIR', BASE_DIR / 'exp'))
DATA_DIRECTION = 'combined'  # combined, Pbgoing, or pgoing
DATA_SELECTION = 'jetId'     # jetId, trkMax, or noSel
DATA_FILES = {
    trigger: resolve_data_file(DATA_DIR, trigger, DATA_DIRECTION, DATA_SELECTION)
    for trigger in ('MinimumBias', 'Jet60', 'Jet80', 'Jet100')
}
PTAVE_BINS = {
    'MinimumBias': [(60, 80), (80, 100), (100, 120), (120, 180)],
    'Jet60': [(80, 100), (100, 120), (120, 180), (180, 250)],
    'Jet80': [(100, 120), (120, 180), (180, 250), (300, 500)],
    'Jet100': [(120, 180), (180, 250), (300, 500)],
}
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.5)
ETA_CUT_INDEX = STANDARD_DIJET_ETA_CUT_INDEX
REBIN_ETA = 2
FORWARD_BACKWARD_RATIO_OPTION = ''  # protected: F / B is never binomial
FULL_COMPARISON_RATIO_OPTION = ''   # '' or 'B' for Up/Def and Down/Def
FB_COMPARISON_RATIO_OPTION = ''     # '' or 'B' for (F/B)_var / (F/B)_Def
FULL_RATIO_RANGE = (0.85, 1.15)
FB_RANGE = (0.75, 1.30)
FB_DOUBLE_RATIO_RANGE = (0.85, 1.15)
SYSTEMATIC_Y_RANGE = None       # percent units, e.g. (0.0, 10.0), or None
APPLY_SYSTEMATIC_SMOOTHING = True
FULL_SMOOTHING_ORIGIN = -0.465 + 0.00001  # CM boost-shifted FindBin point
FULL_FIT_FUNCTION = 'pol2'
FB_FIT_FUNCTION = 'pol1'
FULL_FIT_INITIAL_VALUES = {
    'Up / Def': (1.0, 0.0, 0.0),
    'Down / Def': (1.0, 0.0, 0.0),
}
FB_FIT_INITIAL_VALUES = {
    'Up / Def': (1.0, 0.0),
    'Down / Def': (1.0, 0.0),
}
FIT_OPTIONS = 'RQS0'
SHOW_FIT_RESULTS = True
FIT_RESULTS_TEXT_SIZE = 0.018
FIT_RESULTS_BOX_BOUNDS = (0.43, 0.18, 0.88, 0.40)
DRAW_GRID = True
SAVE_PNG = False
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_JEU_SYSTEMATICS_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'systematics_JEU',
))

CURVES = (
    DijetClosureCurve(
        'JEU Up', 'hRecoDijetPtEtaCMJeuUp_{eta_cut_index}',
        'hRecoDijetPtEtaForwardJeuUp_{eta_cut_index}',
        'hRecoDijetPtEtaBackwardJeuUp_{eta_cut_index}',
    ),
    DijetClosureCurve(
        'JEU Down', 'hRecoDijetPtEtaCMJeuDown_{eta_cut_index}',
        'hRecoDijetPtEtaForwardJeuDown_{eta_cut_index}',
        'hRecoDijetPtEtaBackwardJeuDown_{eta_cut_index}',
    ),
    DijetClosureCurve(
        'JEU Default', 'hRecoDijetPtEtaCM_{eta_cut_index}',
        'hRecoDijetPtEtaForward_{eta_cut_index}',
        'hRecoDijetPtEtaBackward_{eta_cut_index}',
    ),
)
NOMINAL = 'JEU Default'
# Select entries from root_style.COLORS (0 red, 1 blue, 2 black, ...).
HISTOGRAM_COLOR_INDICES = {'JEU Up': 0, 'JEU Down': 1, 'JEU Default': 2}
VARIATION_COLOR_INDICES = {'Up / Def': 0, 'Down / Def': 1}
SYSTEMATIC_COLOR_INDICES = {'JEU bin-by-bin': 3, 'JEU smoothed': 3}
PLOT_STYLE = replace(
    DEFAULT_PLOT_STYLE,
    annotation_text_size=0.026, annotation_line_spacing=0.039,
    legend_text_size=0.028,
)

if set(PTAVE_BINS) != set(DATA_FILES):
    raise ValueError('PTAVE_BINS must define exactly the same samples as DATA_FILES')
for sample, intervals in PTAVE_BINS.items():
    if not intervals:
        raise ValueError(f'PTAVE_BINS[{sample!r}] must not be empty')
    if any(low >= high for low, high in intervals):
        raise ValueError(f'Invalid pTave interval for {sample}: {intervals}')
for sample, filename in DATA_FILES.items():
    if not filename.exists():
        raise FileNotFoundError(f'Missing configured {sample} ROOT file: {filename}')
if ETA_CUT_INDEX < 0 or ETA_CUT_INDEX >= len(ETA_CUTS):
    raise IndexError(f'Invalid eta-cut index: {ETA_CUT_INDEX}')
if FORWARD_BACKWARD_RATIO_OPTION != '':
    raise ValueError('Forward/Backward construction must use standard errors')
for option_name, option in (
    ('FULL_COMPARISON_RATIO_OPTION', FULL_COMPARISON_RATIO_OPTION),
    ('FB_COMPARISON_RATIO_OPTION', FB_COMPARISON_RATIO_OPTION),
):
    if option not in ('', 'B'):
        raise ValueError(f'{option_name} must be empty or B')
for label, color_index in (
    *HISTOGRAM_COLOR_INDICES.items(), *VARIATION_COLOR_INDICES.items(),
    *SYSTEMATIC_COLOR_INDICES.items(),
):
    if not isinstance(color_index, int) or not 0 <= color_index < len(COLORS):
        raise ValueError(f'Invalid root_style color index for {label}: {color_index}')
ETA_CUT = ETA_CUTS[ETA_CUT_INDEX]
DATA_FILES, PTAVE_BINS


## Build projections and systematic comparisons

Each CM projection is scaled by `1 / Integral()` through `normalization='integral'`. Forward and backward projections are not normalized before division. The shared helper is called separately below for every trigger sample.

In [ ]:
def finite_nonzero_range(histogram):
    values = [
        histogram.GetBinContent(index)
        for index in range(1, histogram.GetNbinsX() + 1)
        if histogram.GetBinContent(index) != 0.0
        and math.isfinite(histogram.GetBinContent(index))
    ]
    return (min(values), max(values)) if values else None


def analyze_data_file(sample, input_file):
    results = {}
    eta_x_range = (-ETA_CUT - 0.1, ETA_CUT + 0.1)
    fb_x_range = (0.0, ETA_CUT + 0.1)
    eta_cut_tag = int(round(10.0 * ETA_CUT))
    sample_tag = sample.lower()
    systematic_output_tag = (
        'systematic_smoothed'
        if APPLY_SYSTEMATIC_SMOOTHING else 'systematic_nonsmoothed'
    )

    for ptave_range in PTAVE_BINS[sample]:
        low, high = ptave_range
        ptave_tag = f'{low:g}_{high:g}'.replace('.', 'p')
        common_tag = (
            f'{sample_tag}_jeuSystematics_etaCM_{eta_cut_tag}'
            f'_ptave_{ptave_tag}'
        )
        output_name = lambda plot: (
            f'{sample_tag}_jeuSystematics_{plot}'
            f'_etaCM_{eta_cut_tag}_ptave_{ptave_tag}.pdf'
        )
        eta_shapes, fb_ratios, selected_keys = build_dijet_gen_comparisons(
            input_file, CURVES, eta_cut_index=ETA_CUT_INDEX,
            ptave_range=ptave_range, nominal=NOMINAL, rebin_eta=REBIN_ETA,
            normalization='integral',
            ratio_option=FORWARD_BACKWARD_RATIO_OPTION,
        )
        eta_variations = {
            ratio_label: ratio_to_nominal(
                eta_shapes[source_label], eta_shapes[NOMINAL],
                name=f'h_{common_tag}_{source_label.replace(" ", "_")}_to_default',
                option=FULL_COMPARISON_RATIO_OPTION,
            )
            for ratio_label, source_label in (
                ('Up / Def', 'JEU Up'), ('Down / Def', 'JEU Down'),
            )
        }
        fb_variations = {
            ratio_label: ratio_to_nominal(
                fb_ratios[source_label], fb_ratios[NOMINAL],
                name=f'h_{common_tag}_{source_label.replace(" ", "_")}_fb_to_default',
                option=FB_COMPARISON_RATIO_OPTION,
            )
            for ratio_label, source_label in (
                ('Up / Def', 'JEU Up'), ('Down / Def', 'JEU Down'),
            )
        }
        eta_fit_functions, eta_fit_summaries = fit_histogram_variations(
            eta_variations, formula=FULL_FIT_FUNCTION,
            fit_range=(-ETA_CUT, ETA_CUT),
            name_prefix=f'f_{common_tag}_full_ratio',
            fit_options=FIT_OPTIONS,
            initial_values=FULL_FIT_INITIAL_VALUES,
        )
        fb_fit_functions, fb_fit_summaries = fit_histogram_variations(
            fb_variations, formula=FB_FIT_FUNCTION,
            fit_range=(0.0, ETA_CUT),
            name_prefix=f'f_{common_tag}_fb_ratio',
            fit_options=FIT_OPTIONS,
            initial_values=FB_FIT_INITIAL_VALUES,
        )
        eta_systematic_unsmoothed = calculate_bin_by_bin_systematic(
            eta_variations['Up / Def'], eta_variations['Down / Def'],
            name=f'h_{common_tag}_full_jeu_relative_systematic',
            up_function=eta_fit_functions['Up / Def'],
            down_function=eta_fit_functions['Down / Def'],
            evaluation_range=(-ETA_CUT, ETA_CUT),
        )
        fb_systematic_unsmoothed = calculate_bin_by_bin_systematic(
            fb_variations['Up / Def'], fb_variations['Down / Def'],
            name=f'h_{common_tag}_fb_jeu_relative_systematic',
            up_function=fb_fit_functions['Up / Def'],
            down_function=fb_fit_functions['Down / Def'],
            evaluation_range=(0.0, ETA_CUT),
        )
        eta_systematic = (
            smooth_systematic_running_max(
                eta_systematic_unsmoothed,
                name=f'{eta_systematic_unsmoothed.GetName()}_smoothed',
                evaluation_range=(-ETA_CUT, ETA_CUT),
                smoothing_origin=FULL_SMOOTHING_ORIGIN,
            )
            if APPLY_SYSTEMATIC_SMOOTHING else eta_systematic_unsmoothed
        )
        fb_systematic = (
            smooth_systematic_running_max(
                fb_systematic_unsmoothed,
                name=f'{fb_systematic_unsmoothed.GetName()}_smoothed',
                evaluation_range=(0.0, ETA_CUT),
            )
            if APPLY_SYSTEMATIC_SMOOTHING else fb_systematic_unsmoothed
        )
        systematic_label = (
            'JEU smoothed' if APPLY_SYSTEMATIC_SMOOTHING else 'JEU bin-by-bin'
        )
        eta_systematic_percent = eta_systematic.Clone(
            f'{eta_systematic.GetName()}_percent'
        )
        eta_systematic_percent.SetDirectory(0)
        eta_systematic_percent.Scale(100.0)
        fb_systematic_percent = fb_systematic.Clone(
            f'{fb_systematic.GetName()}_percent'
        )
        fb_systematic_percent.SetDirectory(0)
        fb_systematic_percent.Scale(100.0)
        eta_systematics = {systematic_label: eta_systematic_percent}
        fb_systematics = {systematic_label: fb_systematic_percent}
        eta_fit_text = (
            format_fit_summary_lines(eta_fit_summaries)
            if SHOW_FIT_RESULTS else None
        )
        fb_fit_text = (
            format_fit_summary_lines(fb_fit_summaries)
            if SHOW_FIT_RESULTS else None
        )
        annotations = (
            sample,
            f'{low:g} < p_{{T}}^{{ave}} < {high:g} GeV',
            f'|#eta_{{CM}}^{{jet}}| < {ETA_CUT:g}',
            'p_{T}^{Lead} > 50 GeV',
            'p_{T}^{SubLead} > 40 GeV',
            DIJET_DELTA_PHI_SELECTION_LABEL,
        )
        canvases = {
            'eta_overlay': draw_overlay(
                eta_shapes, title='', x_title='#eta_{CM}^{dijet}',
                y_title='1/N dN/d#eta_{CM}^{dijet}', x_range=eta_x_range,
                annotations=annotations, grid=DRAW_GRID, headroom=1.6,
                style_indices=HISTOGRAM_COLOR_INDICES, style=PLOT_STYLE,
                output=OUTPUT_DIR / output_name('full_overlay'),
                save_png=SAVE_PNG, canvas_name=f'{common_tag}_full_overlay',
            ),
            'eta_variations': draw_overlay(
                eta_variations, title='', x_title='#eta_{CM}^{dijet}',
                y_title='JEU variation / default', x_range=eta_x_range,
                y_range=FULL_RATIO_RANGE, reference_y=1.0,
                annotations=annotations, grid=DRAW_GRID,
                overlay_functions=eta_fit_functions,
                overlay_text=eta_fit_text,
                overlay_text_bounds=FIT_RESULTS_BOX_BOUNDS,
                overlay_text_size=FIT_RESULTS_TEXT_SIZE,
                style_indices=VARIATION_COLOR_INDICES, style=PLOT_STYLE,
                output=OUTPUT_DIR / output_name('full_ratio_to_default'),
                save_png=SAVE_PNG, canvas_name=f'{common_tag}_full_ratio',
            ),
            'fb_overlay': draw_overlay(
                fb_ratios, title='', x_title='#eta_{CM}^{dijet}',
                y_title='Forward / Backward', x_range=fb_x_range,
                y_range=FB_RANGE, annotations=annotations, grid=DRAW_GRID,
                style_indices=HISTOGRAM_COLOR_INDICES, style=PLOT_STYLE,
                output=OUTPUT_DIR / output_name('fb_overlay'),
                save_png=SAVE_PNG, canvas_name=f'{common_tag}_fb_overlay',
            ),
            'fb_variations': draw_overlay(
                fb_variations, title='', x_title='#eta_{CM}^{dijet}',
                y_title='(F/B)_{JEU variation} / (F/B)_{default}',
                x_range=fb_x_range, y_range=FB_DOUBLE_RATIO_RANGE,
                reference_y=1.0, annotations=annotations, grid=DRAW_GRID,
                overlay_functions=fb_fit_functions,
                overlay_text=fb_fit_text,
                overlay_text_bounds=FIT_RESULTS_BOX_BOUNDS,
                overlay_text_size=FIT_RESULTS_TEXT_SIZE,
                style_indices=VARIATION_COLOR_INDICES, style=PLOT_STYLE,
                output=OUTPUT_DIR / output_name('fb_ratio_to_default'),
                save_png=SAVE_PNG, canvas_name=f'{common_tag}_fb_ratio',
            ),
            'eta_systematic': draw_overlay(
                eta_systematics, title='', x_title='#eta_{CM}^{dijet}',
                y_title='JEU Rel. Syst. Uncrt. (%)',
                x_range=eta_x_range, y_range=SYSTEMATIC_Y_RANGE,
                annotations=annotations, grid=DRAW_GRID,
                show_legend=False,
                style_indices=SYSTEMATIC_COLOR_INDICES, style=PLOT_STYLE,
                output=OUTPUT_DIR / output_name(
                    f'{systematic_output_tag}_full_relative'
                ),
                save_png=SAVE_PNG, canvas_name=f'{common_tag}_full_systematic',
            ),
            'fb_systematic': draw_overlay(
                fb_systematics, title='', x_title='#eta_{CM}^{dijet}',
                y_title='JEU Rel. Syst. Uncrt. (%)',
                x_range=fb_x_range, y_range=SYSTEMATIC_Y_RANGE,
                annotations=annotations, grid=DRAW_GRID,
                show_legend=False,
                style_indices=SYSTEMATIC_COLOR_INDICES, style=PLOT_STYLE,
                output=OUTPUT_DIR / output_name(
                    f'{systematic_output_tag}_fb_relative'
                ),
                save_png=SAVE_PNG, canvas_name=f'{common_tag}_fb_systematic',
            ),
        }
        results[ptave_range] = {
            'eta_shapes': eta_shapes, 'eta_variations': eta_variations,
            'forward_backward': fb_ratios, 'fb_variations': fb_variations,
            'eta_systematic': eta_systematic,
            'fb_systematic': fb_systematic,
            'eta_systematic_unsmoothed': eta_systematic_unsmoothed,
            'fb_systematic_unsmoothed': fb_systematic_unsmoothed,
            'systematic_smoothing_applied': APPLY_SYSTEMATIC_SMOOTHING,
            'full_smoothing_origin': FULL_SMOOTHING_ORIGIN,
            'eta_systematic_percent': eta_systematic_percent,
            'fb_systematic_percent': fb_systematic_percent,
            'eta_fit_functions': eta_fit_functions,
            'eta_fit_summaries': eta_fit_summaries,
            'fb_fit_functions': fb_fit_functions,
            'fb_fit_summaries': fb_fit_summaries,
            'keys': selected_keys, 'canvases': canvases,
        }
        print(f'\n{sample}, pTave interval {ptave_range}, eta cut {ETA_CUT:g}')
        print('selected keys:', selected_keys)
        print('CM integral normalization:', {
            label: histogram.Integral()
            for label, histogram in eta_shapes.items()
        })
        print('CM variation/default ranges:', {
            label: finite_nonzero_range(histogram)
            for label, histogram in eta_variations.items()
        })
        print('F/B variation/default ranges:', {
            label: finite_nonzero_range(histogram)
            for label, histogram in fb_variations.items()
        })
        print('Relative systematic ranges:', {
            'CM': finite_nonzero_range(eta_systematic),
            'F/B': finite_nonzero_range(fb_systematic),
        })
        for observable, summaries in (
            ('CM variation/default', eta_fit_summaries),
            ('F/B variation/default', fb_fit_summaries),
        ):
            print(f'{observable} fits:')
            for label, summary in summaries.items():
                coefficients = ', '.join(
                    f'p{index}={value:.6g} +/- {error:.3g}'
                    for index, (value, error) in enumerate(zip(
                        summary['parameters'], summary['parameter_errors'],
                    ))
                )
                print(
                    f"  {label}: {summary['formula']}, {coefficients}, "
                    f"chi2/ndf={summary['chi2']:.3g}/{summary['ndf']}, "
                    f"prob={summary['probability']:.3g}"
                )
        for canvas in canvases.values():
            display(canvas)
    return results


## MinimumBias data

In [ ]:
mb_results = analyze_data_file('MinimumBias', DATA_FILES['MinimumBias'])


## Jet60 data

In [ ]:
jet60_results = analyze_data_file('Jet60', DATA_FILES['Jet60'])


## Jet80 data

In [ ]:
jet80_results = analyze_data_file('Jet80', DATA_FILES['Jet80'])


## Jet100 data

In [ ]:
jet100_results = analyze_data_file('Jet100', DATA_FILES['Jet100'])


In [ ]:
# Overlay ratio-to-default curves with the symmetric evaluated systematic band.
SHOW_FIT_LINES_ON_SYSTEMATIC_BANDS = True  # set False to hide fitted curves
# The retained systematic histograms are fractional: 0.0002 corresponds to 0.02%.
def draw_ratio_with_systematic_band(ratios, systematic, *, x_range, y_range, fit_functions,
                                    output, canvas_name, x_title, annotations):
    canvas = draw_overlay(
        ratios, title='', x_title=x_title, y_title='Variation / default',
        x_range=x_range, y_range=y_range, reference_y=1.0,
        annotations=annotations, grid=DRAW_GRID, overlay_functions=fit_functions,
        style_indices=VARIATION_COLOR_INDICES, style=PLOT_STYLE,
        output=None, save_png=False, canvas_name=canvas_name,
    )
    graph = ROOT.TGraphAsymmErrors(systematic.GetNbinsX())
    for bin_index in range(1, systematic.GetNbinsX() + 1):
        point = bin_index - 1
        graph.SetPoint(point, systematic.GetBinCenter(bin_index), 1.0)
        graph.SetPointError(point, systematic.GetBinWidth(bin_index) / 2.0, systematic.GetBinWidth(bin_index) / 2.0, systematic.GetBinContent(bin_index), systematic.GetBinContent(bin_index))
    graph.SetFillColorAlpha(COLORS[3], 0.30); graph.SetLineColor(COLORS[3]); graph.Draw('E2')
    for ratio in ratios.values(): ratio.Draw('E1 SAME')
    if SHOW_FIT_LINES_ON_SYSTEMATIC_BANDS:
        for function in fit_functions.values(): function.Draw('SAME')
    canvas._overlay_objects[0].AddEntry(graph, 'Syst. Uncrt.', 'f')
    canvas.Modified(); canvas.Update()
    save_canvas(canvas, output, save_png=SAVE_PNG)
    canvas._overlay_objects.append(graph)
    return canvas

eta_x_range = (-ETA_CUT - 0.1, ETA_CUT + 0.1)
fb_x_range = (0.0, ETA_CUT + 0.1)
for sample, sample_results in (('MinimumBias', mb_results), ('Jet60', jet60_results), ('Jet80', jet80_results), ('Jet100', jet100_results)):
    for ptave_range, result in sample_results.items():
        low, high = ptave_range; sample_tag = sample.lower(); ptave_tag = f'{low:g}_{high:g}'.replace('.', 'p')
        eta_cut_tag = int(round(10.0 * ETA_CUT))
        common_tag = f'{sample_tag}_jeuSystematics_etaCM_{eta_cut_tag}_ptave_{ptave_tag}'
        annotations = (sample, f'{low:g} < p_{{T}}^{{ave}} < {high:g} GeV')
        result['eta_ratio_systematic_band'] = draw_ratio_with_systematic_band(result['eta_variations'], result['eta_systematic'], fit_functions=result['eta_fit_functions'], x_range=eta_x_range, y_range=FULL_RATIO_RANGE, output=OUTPUT_DIR / f'{common_tag}_full_ratio_with_systematic_band.pdf', canvas_name=f'{common_tag}_full_ratio_with_systematic_band', x_title='#eta_{CM}^{dijet}', annotations=annotations)
        result['fb_ratio_systematic_band'] = draw_ratio_with_systematic_band(result['fb_variations'], result['fb_systematic'], fit_functions=result['fb_fit_functions'], x_range=fb_x_range, y_range=FB_DOUBLE_RATIO_RANGE, output=OUTPUT_DIR / f'{common_tag}_fb_ratio_with_systematic_band.pdf', canvas_name=f'{common_tag}_fb_ratio_with_systematic_band', x_title='#eta_{CM}^{dijet}', annotations=annotations)
        # Headerless CSV in TGraphErrors order: eta_center, 1.0, eta_half_width, fractional uncertainty.
        for observable, systematic, acceptance in (('full', result['eta_systematic'], (-ETA_CUT, ETA_CUT)), ('fb', result['fb_systematic'], (0.0, ETA_CUT))):
            systematic_output_tag = ('systematic_smoothed' if result['systematic_smoothing_applied'] else 'systematic_nonsmoothed')
            csv_path = OUTPUT_DIR / f'{sample_tag}_jeuSystematics_{systematic_output_tag}_{observable}_relative_etaCM_{eta_cut_tag}_ptave_{ptave_tag}.csv'
            with csv_path.open('w', encoding='utf-8') as stream:
                for bin_index in range(1, systematic.GetNbinsX() + 1):
                    bin_low = systematic.GetXaxis().GetBinLowEdge(bin_index)
                    bin_high = systematic.GetXaxis().GetBinUpEdge(bin_index)
                    if bin_high <= acceptance[0] or bin_low >= acceptance[1]:
                        continue
                    fraction = systematic.GetBinContent(bin_index)
                    stream.write(f'{systematic.GetBinCenter(bin_index):.10g},1.0,{systematic.GetBinWidth(bin_index) / 2.0:.10g},{fraction:.10g}\n')
            result[f'{observable}_systematic_csv'] = csv_path